# Lab 02 - Data Understanding: Global Urban Air Quality and Pollution Time-Series

This notebook applies **Lab 2 - Data Understanding** to the assignment dataset. The goal is to become familiar with the data through dataframe inspection, category counts, and visual exploration.


## Lab 2 concepts used

- Import pandas, seaborn, matplotlib, and warnings.
- Use `head()` and `tail()` to inspect rows.
- Use `value_counts()` to understand important categories or labels.
- Create scatter, jointplot, hue-based scatter, boxplot, stripplot, KDE, pairplot, grouped boxplot, and radviz visualizations where suitable for the dataset.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'AML Assignment' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT


In [ ]:
DATASET_FILENAME = 'global_urban_smog_pm25_hourly.csv'
matches = sorted((PROJECT_ROOT / 'Datasets').glob(f'*/{DATASET_FILENAME}'))
if not matches:
    raise FileNotFoundError(f'Could not find {DATASET_FILENAME} under {PROJECT_ROOT / "Datasets"}')
DATASET_PATH = matches[0]
data = pd.read_csv(DATASET_PATH)
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data['Hazardous_Event_Label'] = data['Hazardous_Event'].map({0: 'Not hazardous', 1: 'Hazardous'})
data.head()


In [ ]:
data.tail()


In [ ]:
print('Shape:', data.shape)
print('Columns:')
for column in data.columns:
    print('-', column)


In [ ]:
print('\nValue counts for City')
display(data['City'].value_counts(dropna=False).head(15))

print('\nValue counts for Hazardous_Event')
display(data['Hazardous_Event'].value_counts(dropna=False).head(15))


## Dataset-adapted Lab 2 visualizations

The original Lab 2 notebook uses Iris features such as sepal length, sepal width, petal length, and species. The cells below keep the same visualization ideas, but use columns that make sense for this dataset.


In [ ]:
selected_cities = data['City'].value_counts().head(8).index
city_df = data[data['City'].isin(selected_cities)].copy()
plot_df = city_df.sample(n=min(8000, len(city_df)), random_state=42)

plot_df.plot(kind='scatter', x='PM2_5_ug_m3', y='European_AQI', alpha=0.25, figsize=(8, 5), title='PM2.5 vs European AQI')
plt.show()

sns.jointplot(x='PM2_5_ug_m3', y='European_AQI', data=plot_df.sample(n=min(4000, len(plot_df)), random_state=42), height=6)
plt.show()

sns.FacetGrid(plot_df, hue='Hazardous_Event_Label', height=6).map(plt.scatter, 'PM2_5_ug_m3', 'European_AQI', alpha=0.35).add_legend()
plt.show()

plt.figure(figsize=(12, 5))
sns.boxplot(x='City', y='PM2_5_ug_m3', data=plot_df)
sns.stripplot(x='City', y='PM2_5_ug_m3', data=plot_df.sample(n=min(1500, len(plot_df)), random_state=42), jitter=True, edgecolor='gray', linewidth=0.3, alpha=0.25)
plt.title('PM2.5 by selected cities')
plt.xticks(rotation=35)
plt.show()

sns.FacetGrid(plot_df, hue='Hazardous_Event_Label', height=6).map(sns.kdeplot, 'PM2_5_ug_m3').add_legend()
plt.show()

pair_cols = ['PM2_5_ug_m3', 'PM10_ug_m3', 'European_AQI', 'UV_Index', 'Hazardous_Event_Label']
sns.pairplot(plot_df[pair_cols].dropna().sample(n=min(2000, len(plot_df)), random_state=42), hue='Hazardous_Event_Label', height=2.2, diag_kind='kde')
plt.show()

box_cols = ['Hazardous_Event_Label', 'PM2_5_ug_m3', 'European_AQI', 'Nitrogen_Dioxide_ug_m3']
plot_df[box_cols].dropna().boxplot(by='Hazardous_Event_Label', figsize=(12, 7))
plt.suptitle('Pollution fields by hazardous event label')
plt.show()

from pandas.plotting import radviz
radviz_cols = ['PM2_5_ug_m3', 'PM10_ug_m3', 'European_AQI', 'UV_Index', 'Hazardous_Event_Label']
radviz(plot_df[radviz_cols].dropna().sample(n=min(1500, len(plot_df)), random_state=42), 'Hazardous_Event_Label')
plt.title('Radviz by hazardous event label')
plt.show()


## What was learned from Lab 2

These plots give an initial view of distributions, category balance, relationships between numeric fields, and possible separation by important labels or groups. No preprocessing or machine learning model is applied yet; that begins from Lab 3 onward.
